In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
scores = pd.read_csv("./genome_scores.csv")
tags = pd.read_csv("./genome_tags.csv")
movies = pd.read_csv("./movie.csv")
ratings = pd.read_csv("./rating.csv")

In [3]:
scores.head()

,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


In [4]:
tags.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [5]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [7]:
# Создаём pivot-таблицу: фильмы по строкам, теги по столбцам, значения — relevance
movie_tag_matrix = scores.pivot_table(index='movieId', columns='tagId', values='relevance', fill_value=0)

In [8]:
# Добавим названия фильмов
movie_tag_matrix = movie_tag_matrix.merge(movies[['movieId', 'title']], on='movieId', how='left')
# Сохраняем названия отдельно, потом вернём их обратно
movie_titles = movie_tag_matrix[['movieId', 'title']]
features = movie_tag_matrix.drop(columns=['movieId', 'title'])

In [9]:
def get_similar_movies(movie_name, top_n=10):
    # Находим индекс фильма
    idx = movie_titles[movie_titles['title'] == movie_name].index[0]

    # Считаем cosine similarity между выбранным фильмом и всеми остальными
    sim_scores = cosine_similarity([features.iloc[idx]], features)[0]

    # Собираем всё в DataFrame
    sim_df = pd.DataFrame({
        'movieId': movie_titles['movieId'],
        'title': movie_titles['title'],
        'similarity': sim_scores
    })

    # Удаляем сам фильм
    sim_df = sim_df[sim_df['title'] != movie_name]

    # Сортируем по убыванию
    recommendations = sim_df.sort_values(by='similarity', ascending=False).head(top_n)

    return recommendations


In [10]:
get_similar_movies('Toy Story (1995)', top_n=10)


,movieId,title,similarity
4331,4886,"Monsters, Inc. (2001)",0.958831
2769,3114,Toy Story 2 (1999),0.955522
2064,2355,"Bug's Life, A (1998)",0.948892
5445,6377,Finding Nemo (2003),0.924944
9070,78499,Toy Story 3 (2010),0.920019
7994,50872,Ratatouille (2007),0.916324
4602,5218,Ice Age (2002),0.914144
3809,4306,Shrek (2001),0.907612
8725,68954,Up (2009),0.904357
2009,2294,Antz (1998),0.900694


In [11]:
# Посчитаем средний рейтинг каждого фильма
avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()

# Добавим в нашу матрицу
movie_data = movie_tag_matrix.merge(avg_ratings, on='movieId', how='left')

# Средние теги у фильмов с рейтингом > 4.0
high_rating_movies = movie_data[movie_data['rating'] > 4.0]
mean_tags = high_rating_movies[features.columns].mean().sort_values(ascending=False)

# Топ-10 тегов
top_tags = mean_tags.head(10)
top_tags = top_tags.reset_index().rename(columns={'index': 'tagId', 0: 'avg_relevance'})
top_tags = top_tags.merge(tags, on='tagId')
top_tags[['tag', 'avg_relevance']]


,tag,avg_relevance
0,imdb top 250,0.866691
1,original,0.787951
2,masterpiece,0.731342
3,oscar (best directing),0.726830
4,criterion,0.725339
5,talky,0.707315
6,great acting,0.692625
7,storytelling,0.678176
8,great ending,0.671409
9,cinematography,0.642847


In [12]:
def recommend_by_multiple_movies(movie_list, top_n=5):
    # Получаем индексы фильмов
    indices = movie_titles[movie_titles['title'].isin(movie_list)].index.tolist()
    
    if len(indices) == 0:
        return "Фильмы не найдены."
    
    # Получаем векторы этих фильмов
    vectors = features.iloc[indices]

    # Вычисляем средний вектор
    mean_vector = vectors.mean(axis=0).values.reshape(1, -1)

    # Считаем схожесть со всеми фильмами
    sim_scores = cosine_similarity(mean_vector, features)[0]

    # Собираем DataFrame с результатами
    sim_df = pd.DataFrame({
        'movieId': movie_titles['movieId'],
        'title': movie_titles['title'],
        'similarity': sim_scores
    })

    # Удаляем фильмы, которые пользователь уже указал
    sim_df = sim_df[~sim_df['title'].isin(movie_list)]

    # Сортируем и возвращаем top_n рекомендаций
    recommendations = sim_df.sort_values(by='similarity', ascending=False).head(top_n)

    return recommendations


In [13]:
my_faves = [
    'Toy Story (1995)',
    'Jumanji (1995)',
    'Grumpier Old Men (1995)',
    'Waiting to Exhale (1995)',
    'Father of the Bride Part II (1995)'
]

recommend_by_multiple_movies(my_faves, top_n=5)


,movieId,title,similarity
1776,2040,"Computer Wore Tennis Shoes, The (1969)",0.909275
2769,3114,Toy Story 2 (1999),0.896116
4090,4621,Look Who's Talking (1989),0.895547
1752,2015,"Absent-Minded Professor, The (1961)",0.893604
5536,6517,"Babe, The (1992)",0.893119
